#### HF Losses in Ferrite Core Cross sections


In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import mph
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

import mphsweepkit as msk

post_processing_exprs = msk.load_post_processing_exprs("post_processing_expressions.json")

post_processing_expressions.json


Start the Comsol Server and load the model

In [4]:
# Start the COMSOL client
client = mph.start()

In [5]:
# Load the model
model = client.load('core_cross_section_initial.mph')

Create a CascadedSweep object

In [11]:
import mphsweepkit as msk
import importlib
importlib.reload(msk)

cs = msk.CascadedSweep(model, 'Study on Cross-Sections')
cs.load_data_from_model()

In [12]:
# cs.sweep_types
# cs.data_unit_map
cs.data

,hor_slit,vert_slit,w,l_r,a_e,matsw.comp1.core,b_mean,freq
0,0.0,0.0,5.0,0.0,5.0,2.0,25.0,100.0
1,0.0,0.0,5.0,0.0,5.0,2.0,25.0,200.0
2,0.0,0.0,5.0,0.0,5.0,2.0,25.0,300.0
3,0.0,0.0,5.0,0.0,5.0,2.0,25.0,400.0
4,0.0,0.0,5.0,0.0,5.0,2.0,25.0,500.0
...,...,...,...,...,...,...,...,...
307,0.0,0.0,15.0,0.0,15.0,3.0,100.0,900.0
308,0.0,0.0,15.0,0.0,15.0,3.0,100.0,1000.0
309,0.0,0.0,15.0,0.0,15.0,3.0,100.0,1100.0
310,0.0,0.0,15.0,0.0,15.0,3.0,100.0,1200.0


In [13]:
# # Set the parametric sweep for the geometric sweep
msk.set_material_sweep(
    sweep_node=cs.sweep_nodes[1],
    material_names=['matsw.comp1.core'],
    material_values=[2.0],
    sweep_type="filled"
)


# print("Updated Material Sweep Parameters:")
# material_sweep_data = get_material_data(material_sweep.properties())
# print_parameters(material_sweep_data)


In [15]:
cs.load_data_from_model()

In [16]:
cs.data

,hor_slit,vert_slit,w,l_r,a_e,matsw.comp1.core,b_mean,freq
0,0.0,0.0,5.0,0.0,5.0,2.0,25.0,100.0
1,0.0,0.0,5.0,0.0,5.0,2.0,25.0,200.0
2,0.0,0.0,5.0,0.0,5.0,2.0,25.0,300.0
3,0.0,0.0,5.0,0.0,5.0,2.0,25.0,400.0
4,0.0,0.0,5.0,0.0,5.0,2.0,25.0,500.0
...,...,...,...,...,...,...,...,...
151,0.0,0.0,15.0,0.0,15.0,2.0,100.0,900.0
152,0.0,0.0,15.0,0.0,15.0,2.0,100.0,1000.0
153,0.0,0.0,15.0,0.0,15.0,2.0,100.0,1100.0
154,0.0,0.0,15.0,0.0,15.0,2.0,100.0,1200.0


In [17]:
cs.simulate()

Directory 'batch_data' already exists. Using existing directory.
Batch directory for the geometric sweep set to: x:\Till_data\repositories\MPhSweepKit\examples\studies\infinite_ferrite_cross_sections\batch_data


Set geometric sweep parameters to get circles with radii: 5, 10, 15 mm

In [ ]:
# # Define a geometric sweep of cylinders with round cross-sections of different radii
# param__radii = np.array([5, 10, 15])  # Radii for the cross-section in mm

# # Create arrays for the parameters based on the radii
# param__hor_slit = np.zeros_like(param__radii)  # Horizontal slit dimensions
# param__vert_slit = np.zeros_like(param__radii) # Vertical slit dimensions
# param__w = param__radii * 2
# param__l_r = np.zeros_like(param__radii)  # Length dimensions
# param__a_e = param__radii

# # Set the parametric sweep for the geometric sweep
# set_parametric_sweep(
#     sweep_obj=geometric_sweep,
#     param_names=["hor_slit", "vert_slit", "w", "l_r", "a_e"],
#     param_units=["um", "um", "mm", "mm", "mm"],
#     param_values=np.array(
#         [param__hor_slit, param__vert_slit, param__w, param__l_r, param__a_e],
#         dtype=float,
#     ),
#     sweep_type="sparse",
# )

# print("Updated Geometric Sweep Parameters:")
# geometric_sweep_data = get_sweep_data(geometric_sweep.properties())
# print_parameters(geometric_sweep_data)

Set material sweep parameters to "N49"

In [ ]:
# # Set the parametric sweep for the geometric sweep
# set_material_sweep(
#     sweep_obj=material_sweep,
#     material_names=['matsw.comp1.core'],
#     material_list=[2.0],
#     sweep_type="filled"
# )


# print("Updated Material Sweep Parameters:")
# material_sweep_data = get_material_data(material_sweep.properties())
# print_parameters(material_sweep_data)

In [ ]:
# # Build the full cascaded dataset in COMSOL sweep order:
# # geometry_sweep -> excitation_sweep -> frequency_sweep.
# df = get_cascaded_dataset(
#     geometric_sweep_data,
#     material_sweep_data,
#     excitation_sweep_data,
#     frequency_sweep_data,
# )

# print(df)

### Evaluate Results - save relevant data/results

In [ ]:
# List all available datasets
datasets = model.datasets()
print(f"Available datasets: {datasets}")

In [ ]:
# Attach all post-processing results to the matching parameter values
for column_name, spec in post_processing_exprs.items():
    df[column_name] = model.evaluate(
        expression=spec["expression"],
        unit=spec["unit"],
        dataset=datasets[-1],
    )

print(df)

Exemplary Visualization of average loss density in the cross-section

In [ ]:
selected_post_processing_key = "p_loss"
selected_post_processing = post_processing_exprs[selected_post_processing_key]


fig, ax = plt.subplots(figsize=(8, 4.5))

# Color map by geometry (w)
w_values = sorted(df["w"].unique())
cmap = plt.get_cmap("viridis", len(w_values))
w_to_color = {w: cmap(i) for i, w in enumerate(w_values)}

# Optional: linestyle by b_mean (helps distinguish curves with same color)
b_values = sorted(df["b_mean"].unique())
linestyles = ["-", "--", "-.", ":"]
b_to_ls = {b: linestyles[i % len(linestyles)] for i, b in enumerate(b_values)}

# Track one curve end per b_mean for labeling (choose highest frequency point)
label_points = {}

for (w_val, b_val), g in df.sort_values("freq").groupby(["w", "b_mean"], sort=True):
    x = g["freq"].to_numpy()
    y = (g[selected_post_processing_key] / 1000).to_numpy()  # kW/m³

    ax.plot(
        x, y,
        marker="o",
        ms=4,
        lw=1.8,
        alpha=0.95,
        color=w_to_color[w_val],      # same color for same w
        linestyle=b_to_ls[b_val],     # optional: style by b_mean
        label=f"w={w_val:.3g} mm"
    )

    # store rightmost point for b_mean label candidates
    i = np.argmax(x)
    xp, yp = x[i], y[i]
    # keep highest y at right edge for this b_mean to avoid overlap a bit
    if b_val not in label_points or (xp >= label_points[b_val][0] and yp > label_points[b_val][1]):
        label_points[b_val] = (xp, yp)

# Axes (datasheet-like)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(f"Frequency (kHz)")
ax.set_ylabel("kW / m³")
ax.set_title(f"{selected_post_processing['label']} over frequency")
ax.grid(True, which="both", alpha=0.3)

# Deduplicated legend for geometry (w)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(),
          title=r"Diameter / Width $w$", bbox_to_anchor=(1.02, 1), loc="upper left")

# Add b_mean text labels near curve ends (datasheet style)
for b_val, (xp, yp) in label_points.items():
    txt = ax.text(
        xp * 1.03, yp, f"{b_val:.3g} mT",
        fontsize=9, va="center", ha="left", color="black"
    )
    # white halo for readability
    txt.set_path_effects([pe.withStroke(linewidth=3, foreground="white")])

# Make room on right for labels
xmin, xmax = ax.get_xlim()
ax.set_xlim(xmin, xmax * 1.25)

plt.tight_layout()
plt.show()# Legend with one entry per w (deduplicate)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    by_label.values(),
    by_label.keys(),
    title=r"Diameter / Width $w$",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

## Saving Models

Save the model with or without solution data:

In [ ]:
# Save to new file
model.save('core_cross_section_solved.mph')

### Release the Client

In [ ]:
client.remove(model)
client.clear()
client.disconnect()